<a href="https://colab.research.google.com/github/liviagaldino/biblioteca-digital-python/blob/main/PROJETO_DE_INVESTIMENTOS_EM_SA%C3%9ADE_NO_ESTADO_DO_PARAN%C3%81_EM_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from google.colab import files

uploaded = files.upload()


Saving dados_saude_pr_24.csv to dados_saude_pr_24 (1).csv


In [24]:
from google.colab import files

uploaded = files.upload()

Saving geojs-41-mun.json to geojs-41-mun (1).json


In [26]:
import pandas as pd
import json
import folium


df_saude = pd.read_csv('dados_saude_pr_24.csv', encoding='utf-8')


with open('geojs-41-mun.json', 'r', encoding='utf-8') as f:
    geo_json_data = json.load(f)


print("Colunas do CSV:", df_saude.columns)
print(df_saude.head())

Colunas do CSV: Index(['Unnamed: 0', 'CO_MUNICIPIO_IBGE', 'MUNICIPIO', 'Valor Liquido'], dtype='object')
  Unnamed: 0  CO_MUNICIPIO_IBGE              MUNICIPIO  Valor Liquido
0          1           410010.0                ABATIA      1718360.90
1          2           410020.0          ADRIANOPOLIS      1613419.75
2          3           410030.0         AGUDOS DO SUL      2615069.78
3          4           410040.0   ALMIRANTE TAMANDARE      8977539.53
4          5           410045.0    ALTAMIRA DO PARANA      2389731.27


In [23]:
import pandas as pd
import json
import folium
import copy
import branca.colormap as cm

# 1. CARREGAR OS DADOS
try:
    df_saude = pd.read_csv('dados_saude_pr_24.csv', encoding='utf-8')
    with open('geojs-41-mun.json', 'r', encoding='utf-8') as f:
        geo_json_data = json.load(f)
    print("✅ Arquivos carregados com sucesso!")
except FileNotFoundError:
    print("❌ Erro: Certifique-se de que os arquivos estão na lateral esquerda do Colab.")
    raise

# 2. TRATAMENTO DOS DADOS DO CSV
coluna_id = 'CO_MUNICIPIO_IBGE'
coluna_valor = 'Valor Liquido'

df_saude = df_saude.dropna(subset=[coluna_id])
df_saude[coluna_id] = df_saude[coluna_id].astype(int).astype(str).str.strip()

# Criar o dicionário de mapeamento
dados_mapeados = df_saude.set_index(coluna_id)[coluna_valor].to_dict()

# 3. CRIAR UMA ESCALA DE CORES CONTÍNUA E SUAVE (Amarelo -> Laranja -> Vermelho)
# Usamos o percentil 95 para evitar que Curitiba estoure a escala sozinha
valor_min = df_saude[coluna_valor].min()
valor_corte = df_saude[coluna_valor].quantile(0.95) # Deixa a escala dinâmica para o interior

escala_cores = cm.LinearColormap(
    colors=['#ffffcc', '#ffb366', '#ee3b3b'], # Amarelo claro, Laranja, Vermelho vivo
    index=[valor_min, (valor_min + valor_corte)/2, valor_corte],
    vmin=valor_min,
    vmax=valor_corte
)
escala_cores.caption = "Gradação de Investimento em Saúde (Até Junho/2024)"

# 4. AJUSTAR O GEOJSON
geo_json_6digitos = copy.deepcopy(geo_json_data)

for feature in geo_json_6digitos['features']:
    id_original = str(feature.get('id', feature['properties'].get('id', ''))).strip()
    id_6digitos = id_original[:-1]

    feature['id'] = id_6digitos

    valor_investido = dados_mapeados.get(id_6digitos, 0)
    feature['properties']['investimento_num'] = float(valor_investido)
    feature['properties']['investimento_str'] = f"R$ {float(valor_investido):,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

# 5. CRIAR O MAPA INTERATIVO (FOLIUM)
lat_pr, lon_pr = -24.6, -51.5
mapa = folium.Map(location=[lat_pr, lon_pr], zoom_start=7.4, tiles="cartodbpositron")

# Adiciona a legenda de cores contínua ao mapa
escala_cores.add_to(mapa)

# Renderizar os municípios com o gradiente contínuo
folium.GeoJson(
    geo_json_6digitos,
    style_function=lambda feature: {
        'fillColor': escala_cores(min(feature['properties']['investimento_num'], valor_corte)), # Limita o teto no corte para Curitiba não apagar o resto
        'color': '#666666',     # Limites dos municípios bem definidos (Exigência da rubrica)
        'weight': 0.6,
        'fillOpacity': 0.8
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'investimento_str'],
        aliases=['Município:', 'Investimento:'],
        localize=True,
        sticky=True
    )
).add_to(mapa)

folium.LayerControl().add_to(mapa)

# Salvar o mapa em formato HTML
nome_arquivo_html = "mapa_interativo_saude_pr.html"
mapa.save(nome_arquivo_html)
print(f"🎉 Novo mapa gerado com gradiente contínuo perfeito!")

# 6. DOWNLOAD AUTOMÁTICO
from google.colab import files
files.download(nome_arquivo_html)

✅ Arquivos carregados com sucesso!
🎉 Novo mapa gerado com gradiente contínuo perfeito!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>